# ADM and BSSN Conversions

Convert a simple ADM data set to BSSN variables and check an inverse conversion on a toy metric.

Navigation: [Index](../index.ipynb) | Previous: [ADM 3+1 Background](adm_3plus1_background.ipynb) | Next: [Index](../index.ipynb)


## Learning Goals

- Convert ADM variables into BSSN variables used for stable numerical evolution.
- Understand the conformal factor as a separated scale of the spatial metric.
- Check that converting to BSSN and back preserves a toy data set.

## Words for This Notebook

- **BSSN:** a rearranged set of Einstein-equation variables commonly used in stable numerical relativity codes.
- **Conformal factor:** the scale pulled out of the spatial metric.
- **Round trip:** convert from one description to another and then back.
- **Residual:** zero means the round trip preserved the original quantity.

Use the code cells actively: first predict what should happen, then run the cell, then explain the output in plain language. This predict-run-explain pattern keeps the physics idea connected to the programming details.


## Convert ADM Variables and Check a Toy Inverse
The conformal factor records the scale removed from the spatial metric. The residual check verifies that a conformally flat toy metric survives an ADM-to-BSSN-to-ADM round trip.

## Import SymPy for Round-Trip Checks

These imports expose the NRPy and Python tools used in the next steps.


In [1]:
import sympy as sp


## Import ADM/BSSN Conversion Tools

These imports expose the NRPy registries and infrastructure writers used below.


In [2]:
import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.general_relativity.ADM_to_BSSN import ADM_to_BSSN
from nrpy.equations.general_relativity.BSSN_to_ADM import BSSN_to_ADM
from nrpy.equations.general_relativity.InitialData_Cartesian import (
    InitialData_Cartesian,
)


## Step 3: Choose the conformal-factor variable

The setting selects which BSSN conformal variable the conversion routines use.

In [3]:
par.set_parval_from_str("EvolvedConformalFactor_cf", "W")
print("conformal factor choice:", par.parval_from_str("EvolvedConformalFactor_cf"))


conformal factor choice: W


## Step 4: Convert Brill-Lindquist ADM data

The printed BSSN variables show the conformal decomposition of one analytic ADM data set.

In [4]:
brill_lindquist = InitialData_Cartesian("BrillLindquist")
adm_to_bssn = ADM_to_BSSN(
    brill_lindquist.gammaDD,
    brill_lindquist.KDD,
    brill_lindquist.betaU,
    brill_lindquist.BU,
    "Cartesian",
)
print("cf:", adm_to_bssn.cf)
print("hDD[0][0]:", adm_to_bssn.hDD[0][0])
print("hDD[0][1]:", adm_to_bssn.hDD[0][1])
print("aDD[0][0]:", adm_to_bssn.aDD[0][0])
print("trK:", adm_to_bssn.trK)
print("vetU:", adm_to_bssn.vetU)
print("betU:", adm_to_bssn.betU)


Setting up reference_metric[Cartesian]...
cf: ((BH1_mass/(2*sqrt((-BH1_posn_x + x)**2 + (-BH1_posn_y + y)**2 + (-BH1_posn_z + z)**2)) + BH2_mass/(2*sqrt((-BH2_posn_x + x)**2 + (-BH2_posn_y + y)**2 + (-BH2_posn_z + z)**2)) + 1)**12)**(-1/6)
hDD[0][0]: (BH1_mass/(2*sqrt((-BH1_posn_x + x)**2 + (-BH1_posn_y + y)**2 + (-BH1_posn_z + z)**2)) + BH2_mass/(2*sqrt((-BH2_posn_x + x)**2 + (-BH2_posn_y + y)**2 + (-BH2_posn_z + z)**2)) + 1)**4*((BH1_mass/(2*sqrt((-BH1_posn_x + x)**2 + (-BH1_posn_y + y)**2 + (-BH1_posn_z + z)**2)) + BH2_mass/(2*sqrt((-BH2_posn_x + x)**2 + (-BH2_posn_y + y)**2 + (-BH2_posn_z + z)**2)) + 1)**(-12))**(1/3) - 1
hDD[0][1]: 0
aDD[0][0]: 0
trK: 0
vetU: [0, 0, 0]
betU: [0, 0, 0]


## Step 5: Build a conformally flat toy metric

This small metric makes the ADM-to-BSSN conversion easy to check by hand.

In [5]:
psi = sp.symbols("psi", positive=True)
toy_gammaDD = ixp.zerorank2()
toy_KDD = ixp.zerorank2()
toy_betaU = ixp.zerorank1()
toy_BU = ixp.zerorank1()
for i in range(3):
    toy_gammaDD[i][i] = psi**4


## Step 6: Round-trip the toy data

The residuals confirm that converting to BSSN and back preserves the original ADM data.

In [6]:
toy_adm_to_bssn = ADM_to_BSSN(toy_gammaDD, toy_KDD, toy_betaU, toy_BU, "Cartesian")
toy_bssn_to_adm = BSSN_to_ADM("Cartesian")
substitutions = {}
generic_expressions = []
for i in range(3):
    generic_expressions.extend(toy_bssn_to_adm.gammaDD[i])
    generic_expressions.extend(toy_bssn_to_adm.KDD[i])
generic_expressions.extend(toy_bssn_to_adm.betaU)
for expression in generic_expressions:
    for symbol in expression.free_symbols:
        name = symbol.name
        if name.startswith("hDD") or name.startswith("aDD") or name == "trK":
            substitutions[symbol] = 0
        elif name == "cf":
            substitutions[symbol] = toy_adm_to_bssn.cf
        elif name.startswith("vetU") or name.startswith("betU"):
            substitutions[symbol] = 0


Setting up BSSN_Quantities[Cartesian]...


## Verify the ADM Round Trip

A zero residual confirms that the symbolic expression matches the expected identity.


In [7]:
metric_residuals = [
    sp.factor(toy_bssn_to_adm.gammaDD[i][i].xreplace(substitutions) - toy_gammaDD[i][i])
    for i in range(3)
]
curvature_residuals = [
    sp.factor(toy_bssn_to_adm.KDD[i][i].xreplace(substitutions) - toy_KDD[i][i])
    for i in range(3)
]
shift_residuals = [
    sp.factor(toy_bssn_to_adm.betaU[i].xreplace(substitutions) - toy_betaU[i])
    for i in range(3)
]
print("toy cf:", toy_adm_to_bssn.cf)
print("toy hDD[0][0]:", toy_adm_to_bssn.hDD[0][0])
print("toy aDD[0][0]:", toy_adm_to_bssn.aDD[0][0])
print("toy trK:", toy_adm_to_bssn.trK)
print("metric diagonal residuals:", metric_residuals)
print("KDD diagonal residuals:", curvature_residuals)
print("shift residuals:", shift_residuals)
if (
    metric_residuals != [0, 0, 0]
    or curvature_residuals != [0, 0, 0]
    or shift_residuals != [0, 0, 0]
):
    raise RuntimeError("Expected all ADM-BSSN round-trip residuals to vanish.")


toy cf: psi**(-2)
toy hDD[0][0]: 0
toy aDD[0][0]: 0
toy trK: 0
metric diagonal residuals: [0, 0, 0]
KDD diagonal residuals: [0, 0, 0]
shift residuals: [0, 0, 0]


The determinant and reconstruction residuals confirm that the ADM-to-BSSN conversion preserves the represented geometry in this example. The same consistency checks matter before generating evolution code.


## Learning Check

Before the round-trip check, predict what should happen to the metric residuals if the conversion formulas are consistent. Then explain why zero residuals matter.


## Finish the Introductory Route
- [ADM 3+1 Background](adm_3plus1_background.ipynb)
- [Indexed Expressions](../1-intro/indexedexp.ipynb)
- [Basis Transforms](../4-curvilinear/basis_transforms.ipynb)
